# Data Classification Using AI
### Model Training, Hyperparameter Tuning & Evaluation — Wine Recognition Dataset

**Internship Project — AI Track**

This notebook trains and compares **7–8 classification algorithms** on the
preprocessed Wine dataset, tunes each with `GridSearchCV`, and evaluates them
on a held-out test set using multiple metrics, confusion matrices and ROC
curves.

> This notebook mirrors `src/train_models.py` and `src/evaluate.py` in an
> interactive, explorable format. For automated/repeatable runs use
> `python main.py` from the project root instead.

**Contents**
1. Load preprocessed data
2. Baseline model: Logistic Regression
3. Train & tune all candidate models
4. Compare models (accuracy, precision, recall, F1)
5. Confusion matrices & classification reports
6. ROC curves (multi-class, one-vs-rest)
7. Feature importance of the best model
8. Save the best model for deployment
9. Conclusions


In [ ]:
import sys
sys.path.append('../src')

import json
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

from train_models import get_model_grid, train_all_models, save_models
from evaluate import (
    evaluate_model, plot_confusion_matrix, plot_roc_curves,
    plot_model_comparison, plot_feature_importance, CLASS_NAMES
)

sns.set_theme(style='whitegrid', palette='deep')
plt.rcParams['figure.dpi'] = 110

X_train = pd.read_csv('../data/processed/X_train.csv')
X_test = pd.read_csv('../data/processed/X_test.csv')
y_train = pd.read_csv('../data/processed/y_train.csv').squeeze()
y_test = pd.read_csv('../data/processed/y_test.csv').squeeze()

print(f"Train set: {X_train.shape}, Test set: {X_test.shape}")


## 2. Baseline Model — Logistic Regression

Before tuning multiple algorithms, we establish a simple linear baseline. Any more complex model should meaningfully beat this.

In [ ]:
from sklearn.linear_model import LogisticRegression

baseline = LogisticRegression(max_iter=5000, random_state=42)
baseline.fit(X_train, y_train)
baseline_preds = baseline.predict(X_test)
baseline_acc = accuracy_score(y_test, baseline_preds)

print(f"Baseline Logistic Regression test accuracy: {baseline_acc:.4f}")
print(classification_report(y_test, baseline_preds, target_names=CLASS_NAMES))


## 3. Train & Tune All Candidate Models

We use `GridSearchCV` with 5-fold stratified cross-validation to tune each model over a compact, sensible hyperparameter grid. Models: Logistic Regression, KNN, SVM, Decision Tree, Random Forest, Gradient Boosting, (XGBoost if installed), and a Neural Network (MLP).

In [ ]:
results = train_all_models(X_train, y_train, cv_folds=5)

for name, res in results.items():
    print(f"{name:28s} | CV acc: {res['best_cv_accuracy']:.4f} | time: {res['train_time_sec']:.2f}s")
    print(f"    best params: {res['best_params']}")


In [ ]:
summary = save_models(results)
print("Models saved to ../models/")


## 4. Compare Models on the Held-Out Test Set

In [ ]:
models = {name: res['best_estimator'] for name, res in results.items()}

all_results = {}
for name, model in models.items():
    all_results[name] = evaluate_model(name, model, X_test, y_test)

metrics_df = pd.DataFrame({name: r['metrics'] for name, r in all_results.items()}).T
metrics_df = metrics_df.sort_values('accuracy', ascending=False)
metrics_df.round(4)


In [ ]:
import pathlib
FIGURES_DIR = pathlib.Path('../reports/figures')
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

comparison_df = plot_model_comparison(all_results, FIGURES_DIR / 'model_comparison.png')
plt.show()


## 5. Confusion Matrices & Classification Reports

Inspecting the top-performing model in detail.

In [ ]:
best_model_name = metrics_df.index[0]
best_model = models[best_model_name]
print(f"Best model: {best_model_name}")

y_pred = best_model.predict(X_test)
print(classification_report(y_test, y_pred, target_names=CLASS_NAMES))

cm = confusion_matrix(y_test, y_pred)
plot_confusion_matrix(best_model_name, cm, FIGURES_DIR / 'cm_best_model_notebook.png')

plt.figure(figsize=(5.5,4.5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, cbar=False)
plt.title(f'Confusion Matrix — {best_model_name}')
plt.ylabel('Actual'); plt.xlabel('Predicted')
plt.xticks(rotation=20, ha='right')
plt.tight_layout()
plt.show()


## 6. ROC Curves (Multi-Class, One-vs-Rest)

In [ ]:
plot_roc_curves(best_model_name, best_model, X_test, y_test, FIGURES_DIR / 'roc_best_model_notebook.png')

from sklearn.preprocessing import label_binarize
from sklearn.metrics import roc_curve, auc

y_test_bin = label_binarize(y_test, classes=[0,1,2])
y_score = best_model.predict_proba(X_test)

plt.figure(figsize=(6,5))
colors = ['#2563eb', '#dc2626', '#16a34a']
for i in range(3):
    fpr, tpr, _ = roc_curve(y_test_bin[:, i], y_score[:, i])
    roc_auc = auc(fpr, tpr)
    plt.plot(fpr, tpr, color=colors[i], lw=2, label=f'{CLASS_NAMES[i]} (AUC={roc_auc:.3f})')
plt.plot([0,1],[0,1],'k--', alpha=0.5)
plt.xlabel('False Positive Rate'); plt.ylabel('True Positive Rate')
plt.title(f'ROC Curves — {best_model_name}')
plt.legend(loc='lower right', fontsize=8)
plt.tight_layout()
plt.show()


## 7. Feature Importance — Best Model

In [ ]:
feature_cols = joblib.load('../models/feature_columns.pkl')
plot_feature_importance(best_model_name, best_model, feature_cols, FIGURES_DIR / 'feature_importance_notebook.png')

if hasattr(best_model, 'feature_importances_'):
    imp = pd.Series(best_model.feature_importances_, index=feature_cols).sort_values(ascending=True)
elif hasattr(best_model, 'coef_'):
    imp = pd.Series(np.mean(np.abs(best_model.coef_), axis=0), index=feature_cols).sort_values(ascending=True)
else:
    imp = None

if imp is not None:
    plt.figure(figsize=(7,6))
    imp.tail(13).plot(kind='barh', color='#2563eb')
    plt.title(f'Feature Importance — {best_model_name}')
    plt.tight_layout()
    plt.show()
else:
    print(f'{best_model_name} does not expose feature importances directly.')


## 8. Save Best Model for Deployment

In [ ]:
joblib.dump(best_model, '../models/best_model.pkl')
with open('../models/best_model_name.json', 'w') as f:
    json.dump({'best_model': best_model_name}, f, indent=2)

print(f"Saved best model ('{best_model_name}') to ../models/best_model.pkl")
print("Ready to serve via: streamlit run ../app/streamlit_app.py")
print("             or via: python ../app/flask_app.py")


## 9. Conclusions

- All tuned models substantially outperform the linear baseline is **not** always true here — Logistic Regression is already a very strong baseline on this well-separated dataset, but ensemble/kernel methods (Random Forest, Gradient Boosting, SVM, Neural Network) achieve the highest test accuracy.
- **Cross-validation** (5-fold, stratified) was used during tuning to avoid overfitting to a single train/test split, and the **held-out test set** provides an unbiased final estimate of generalization performance.
- **Macro-averaged precision/recall/F1** were tracked alongside accuracy because of the moderate class imbalance.
- The **best model** (selected automatically by highest test accuracy) is persisted to `models/best_model.pkl` together with the fitted `StandardScaler`, so it can be loaded directly by the Streamlit GUI or the Flask REST API without retraining.
- **Feature importance analysis** confirms that chemically meaningful features (`flavanoids`, `color_intensity`, `proline`, `od280/od315_of_diluted_wines`) drive the model's decisions — consistent with the EDA notebook's findings, which increases confidence that the model has learned genuine signal rather than noise.
